In [1]:

import os
import requests
import pandas as pd
from dotenv import load_dotenv

# Cargar variables de entorno
load_dotenv()

BASE_URL = os.getenv(
    "WORLD_BANK_API_URL",
    "https://api.worldbank.org/v2"
)

# Indicadores a descargar
indicadores = {
    "SP.POP.TOTL": "poblacion",
    "NY.GDP.MKTP.CD": "pib",
    "FP.CPI.TOTL.ZG": "inflacion",
    "SL.UEM.TOTL.ZS": "desempleo",
    "SP.DYN.LE00.IN": "esperanza_vida",
    "EG.USE.PCAP.KG.OE": "energia_per_capita",
    "IT.NET.USER.ZS": "usuarios_internet",
    "SE.XPD.TOTL.GD.ZS": "gasto_educacion",
    "SH.XPD.CHEX.GD.ZS": "gasto_salud"
}


def obtener_indicador(codigo, nombre):
    """Descarga todas las páginas de un indicador."""

    url = f"{BASE_URL}/country/all/indicator/{codigo}"
    registros = []
    pagina = 1

    try:
        while True:
            parametros = {
                "format": "json",
                "per_page": 20000,
                "page": pagina
            }

            respuesta = requests.get(
                url,
                params=parametros,
                timeout=60
            )

            # Detectar errores HTTP (404, 500, etc.)
            respuesta.raise_for_status()

            datos = respuesta.json()

            if not isinstance(datos, list) or len(datos) < 2:
                print(f"Respuesta inválida para {codigo}")
                return None

            metadata, resultados = datos

            if not isinstance(resultados, list):
                print(f"Sin datos válidos para {codigo}")
                return None

            registros.extend(resultados)

            total_paginas = int(metadata.get("pages", 1))

            if pagina >= total_paginas:
                break

            pagina += 1

        if not registros:
            print(f"No hay registros para {codigo}")
            return None

        # Convertir JSON a DataFrame
        df = pd.json_normalize(registros)

        columnas = ["countryiso3code", "date", "value"]

        if not all(col in df.columns for col in columnas):
            print(f"Columnas faltantes en {codigo}")
            return None

        # Seleccionar y renombrar columnas
        df = df[columnas].copy()

        df.columns = [
            "country_code",
            "year",
            nombre
        ]

        # Corregir tipos de datos
        df["year"] = pd.to_numeric(
            df["year"],
            errors="coerce"
        )

        df[nombre] = pd.to_numeric(
            df[nombre],
            errors="coerce"
        )

        # Eliminar registros sin país o año
        df = df.dropna(
            subset=["country_code", "year"]
        )

        df = df[df["country_code"].str.strip() != ""]

        df["year"] = df["year"].astype(int)

        # Eliminar valores nulos del indicador
        df = df.dropna(subset=[nombre])

        # Evitar duplicados en la unión
        df = df.drop_duplicates(
            subset=["country_code", "year"]
        )

        print(f"{nombre}: {len(df)} registros obtenidos")

        return df

    except requests.exceptions.RequestException as error:
        print(f"Error HTTP o de conexión en {codigo}: {error}")
        return None

    except (ValueError, TypeError, KeyError) as error:
        print(f"Error procesando {codigo}: {error}")
        return None


def main():
    dfs = []

    # Descargar indicadores
    for codigo, nombre in indicadores.items():

        print(f"\nDescargando {nombre} ({codigo})...")

        df_temp = obtener_indicador(codigo, nombre)

        if df_temp is not None:
            dfs.append(df_temp)

    # Comprobar que haya datos
    if not dfs:
        print("No se pudieron descargar los indicadores.")
        return

    # Unir datasets por país y año
    df_final = dfs[0]

    for df_temp in dfs[1:]:
        df_final = pd.merge(
            df_final,
            df_temp,
            on=["country_code", "year"],
            how="outer",
            validate="one_to_one"
        )

    # Ordenar datos
    df_final = df_final.sort_values(
        by=["country_code", "year"]
    )

    # Exportar a CSV
    archivo_salida = "data_extracted.csv"

    try:
        df_final.to_csv(
            archivo_salida,
            index=False,
            encoding="utf-8-sig"
        )

        print("\nArchivo CSV generado correctamente.")
        print(f"Archivo: {archivo_salida}")
        print(f"Filas: {len(df_final)}")
        print(f"Columnas: {len(df_final.columns)}")

        print("\nPrimeros registros:")
        print(df_final.head())

    except OSError as error:
        print(f"Error guardando el CSV: {error}")


if __name__ == "__main__":
    main()



Descargando poblacion (SP.POP.TOTL)...
poblacion: 17130 registros obtenidos

Descargando pib (NY.GDP.MKTP.CD)...
pib: 14481 registros obtenidos

Descargando inflacion (FP.CPI.TOTL.ZG)...
inflacion: 11289 registros obtenidos

Descargando desempleo (SL.UEM.TOTL.ZS)...
desempleo: 8036 registros obtenidos

Descargando esperanza_vida (SP.DYN.LE00.IN)...
esperanza_vida: 16866 registros obtenidos

Descargando energia_per_capita (EG.USE.PCAP.KG.OE)...
energia_per_capita: 6551 registros obtenidos

Descargando usuarios_internet (IT.NET.USER.ZS)...
usuarios_internet: 6812 registros obtenidos

Descargando gasto_educacion (SE.XPD.TOTL.GD.ZS)...
gasto_educacion: 6251 registros obtenidos

Descargando gasto_salud (SH.XPD.CHEX.GD.ZS)...
gasto_salud: 5596 registros obtenidos

Archivo CSV generado correctamente.
Archivo: data_extracted.csv
Filas: 17130
Columnas: 11

Primeros registros:
  country_code  year  poblacion  pib  inflacion  desempleo  esperanza_vida  \
0          ABW  1960    54922.0  NaN     